In [ ]:
#import libraries
import pandas as pd
from sklearn.cluster import KMeans
import numpy as np
from permetrics import ClusteringMetric #source: https://permetrics.readthedocs.io/en/latest/pages/clustering/PuS.html
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import normalize
from gensim.models import KeyedVectors
import urllib.request
from pathlib import Path
import sys
from sklearn.neighbors import NearestNeighbors
from numpy.linalg import norm
import scipy.stats as stats

#We load the floret embeddings
floret_embeddings = KeyedVectors.load_word2vec_format('floret_embeddings_ca.vec')

#We load the fasttext embeddings
#Source: https://zenodo.org/records/4522041 
path = Path("C:/Users/mirei/OneDrive/Escritorio/Uni/Master/TFM/catalan-word-embeddings-fasttext")
ft_skipg_300 = KeyedVectors.load_word2vec_format(str(path / "skipgram" / "d300" / "cat.vec"))


In [ ]:
#import NRC VAD Lexicon 
#source = https://saifmohammad.com/WebPages/nrc-vad.html 

df = pd.read_csv("NRC-VAD-Lexicon-ForVariousLanguages.txt", sep="\t")
#We filter out all the other languages, keeping English as reference 
df = df[['English Word', 'Valence', 'Arousal', 'Dominance', "Catalan"]]
#Normalize 
df['Catalan'] = df['Catalan'].str.lower()

#We create function to evaluate our bias 
def valence_bias(embeddings, model_name, target):
    print(f'Model {model_name}')
    print('-'*100)
    #We get all words but making sure that the words are in the NRC VAD Lexicon 
    all_words = list(embeddings.index_to_key) #these are the words in the embeddings
    catalan_words = set(df['Catalan']) #these are the words in the NRC VAD Lexicon for Catalan
    words_in_NRC = [w for w in all_words if w in catalan_words] #these are the match of both

    #We get the vectors 
    vecs_lexicon = np.array([embeddings[w] for w in words_in_NRC])

    # We start with just one target word 
    target_word = target
    target_vec = embeddings[target_word]  

    #source = https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html 
    # We get the nearest neighbors using similarity (replicating Mendelsohn, Tsvetkov and Jurafsky) using NearestNeighbors
    neigh = NearestNeighbors(n_neighbors=200, metric='cosine')
    neigh.fit(vecs_lexicon)

    neigh_10 = NearestNeighbors(n_neighbors=10, metric='cosine') 
    neigh_10.fit(vecs_lexicon)

    #Here we compute the 200 nearest neighbors, we put the words in a list 
    neighbors = neigh.kneighbors([target_vec], return_distance=False)[0]
    nearest_200 = [words_in_NRC[i] for i in neighbors]

    #And here the 10 nearest neighbors  
    neighbors_10 = neigh_10.kneighbors([target_vec], return_distance=False)[0]
    nearest_10 = [words_in_NRC[i] for i in neighbors_10]


    #We print the valence of the top 10 neighbours 
    print(f"Valence of top 10 nearest neighbours for '{target}' in NRC VAD Lexicon: ")
    print()
    for w in nearest_10:
        match = df[df['Catalan'] == w]
        if not match.empty:
            print(f"Valence of {w}: {match['Valence'].values[0]}")

    #Now we compute the mean average valence of all the 200 nearest neighbors 
    valence = []
    for w in nearest_200:
        match = df[df['Catalan'] == w]
        if not match.empty:
            valence.append(match['Valence'].values[0])
    mean = sum(valence)/len(valence)
    print('-'*100)
    print(f'Mean average valence of 200 neighbours of {target} in NRC VAD Lexicon: {mean:.4f}')
    print('='*100)

#We create a function to evaluate the dominance
def dominance_bias(embeddings, model_name, target):
    print(f'Model {model_name}')
    #We get all words but making sure that the words are in the NRC VAD Lexicon 
    all_words = list(embeddings.index_to_key) #these are the words in the embeddings
    catalan_words = set(df['Catalan']) #these are the words in the NRC VAD Lexicon for Catalan
    words_in_NRC = [w for w in all_words if w in catalan_words] #these are the match of both

    #We get the vectors 
    vecs_lexicon = np.array([embeddings[w] for w in words_in_NRC])

    # We start with just one target word 
    target_word = target
    target_vec = embeddings[target_word]  

    #source = https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html 
    # We get the nearest neighbors using similarity (replicating Mendelsohn, Tsvetkov and Jurafsky) using NearestNeighbors
    neigh = NearestNeighbors(n_neighbors=200, metric='cosine')
    neigh.fit(vecs_lexicon)

    neigh_10 = NearestNeighbors(n_neighbors=10, metric='cosine') 
    neigh_10.fit(vecs_lexicon)

    #Here we compute the 200 nearest neighbors, we put the words in a list 
    neighbors = neigh.kneighbors([target_vec], return_distance=False)[0]
    nearest_200 = [words_in_NRC[i] for i in neighbors]

    #And here the 10 nearest neighbors  
    neighbors_10 = neigh_10.kneighbors([target_vec], return_distance=False)[0]
    nearest_10 = [words_in_NRC[i] for i in neighbors_10]


    #We print the dominance score of the top 10 neighbours 
    print(f"Dominance score of top 10 nearest neighbours for '{target}' in NRC VAD Lexicon: ")
    print()
    for w in nearest_10:
        match = df[df['Catalan'] == w]
        if not match.empty:
            print(f"Dominance score of {w}: {match['Dominance'].values[0]}")

    #Now we compute the mean average dominance score of all the 200 nearest neighbors 
    dominance = []
    for w in nearest_200:
        match = df[df['Catalan'] == w]
        if not match.empty:
            dominance.append(match['Dominance'].values[0])
    mean = sum(dominance)/len(dominance)
    print('-'*100)
    print(f'Mean average dominance score of 200 neighbours of {target} in NRC VAD Lexicon: {mean:.4f}')
    print('='*100)

In [33]:
valence_bias(floret_embeddings, "Floret Embeddings", 'cadira')

Model Floret Embeddings
----------------------------------------------------------------------------------------------------
Valence of top 10 nearest neighbours for 'cadira' in NRC VAD Lexicon: 

Valence of cadira: 0.562
Valence of seient: 0.542
Valence of butaca: 0.646
Valence of llit: 0.604
Valence of bicicleta: 0.708
Valence of motxilla: 0.729
Valence of cabina: 0.531
Valence of tauleta: 0.562
Valence of lliscar: 0.561
Valence of sofà: 0.594
----------------------------------------------------------------------------------------------------
Mean average valence of 200 neighbours of cadira in NRC VAD Lexicon: 0.5203


In [34]:
valence_bias(ft_skipg_300, "Skipgram Embeddings", 'cadira')

Model Skipgram Embeddings
----------------------------------------------------------------------------------------------------
Valence of top 10 nearest neighbours for 'cadira' in NRC VAD Lexicon: 

Valence of cadira: 0.562
Valence of butaca: 0.646
Valence of seient: 0.542
Valence of tamboret: 0.448
Valence of rodes: 0.667
Valence of assegut: 0.561
Valence of cotxet: 0.786
Valence of sofà: 0.594
Valence of llitera: 0.438
Valence of reclinar: 0.46
----------------------------------------------------------------------------------------------------
Mean average valence of 200 neighbours of cadira in NRC VAD Lexicon: 0.5146


In [28]:
valence_bias(floret_embeddings, "Floret Embeddings", 'horrible')

Model Floret Embeddings
----------------------------------------------------------------------------------------------------
Valence of top 10 nearest neighbours for 'horrible' in NRC VAD Lexicon: 

Valence of horrible: 0.061
Valence of terrible: 0.092
Valence of desagradable: 0.042
Valence of cruel: 0.122
Valence of brutal: 0.122
Valence of abominable: 0.12
Valence of crueltat: 0.092
Valence of horriblement: 0.05
Valence of miserable: 0.208
Valence of inevitable: 0.316
----------------------------------------------------------------------------------------------------
Mean average valence of 200 neighbours of horrible in NRC VAD Lexicon: 0.2705


In [69]:
dominance_bias(floret_embeddings, "Floret Embeddings", 'persona')

Model Floret Embeddings
Dominance score of top 10 nearest neighbours for 'persona' in NRC VAD Lexicon: 

Dominance score of persona: 0.596
Dominance score of personalitat: 0.79
Dominance score of circumstància: 0.596
Dominance score of personal: 0.587
Dominance score of individualitat: 0.527
Dominance score of sexualitat: 0.657
Dominance score of conducta: 0.75
Dominance score of dona: 0.398
Dominance score of professió: 0.786
Dominance score of criatura: 0.62
----------------------------------------------------------------------------------------------------
Mean average dominance score of 200 neighbours of persona in NRC VAD Lexicon: 0.5640


In [68]:
dominance_bias(ft_skipg_300, "Skipgram Embeddings", 'persona')

Model Skipgram Embeddings
Dominance score of top 10 nearest neighbours for 'persona' in NRC VAD Lexicon: 

Dominance score of persona: 0.596
Dominance score of dona: 0.398
Dominance score of gent: 0.464
Dominance score of persones: 0.588
Dominance score of noia: 0.433
Dominance score of criatura: 0.62
Dominance score of nena: 0.424
Dominance score of pacient: 0.34
Dominance score of víctima: 0.132
Dominance score of treballador: 0.684
----------------------------------------------------------------------------------------------------
Mean average dominance score of 200 neighbours of persona in NRC VAD Lexicon: 0.5638


In [66]:
#We create a function for moral disgust 
def disgust_bias(embeddings):
    #We create a list of moral disgust words, adapted from the Moral Foundations Theory
    #We utilize the lexicons created by Graham et al. (2009), translated by Google Translate to Catalan
    disgust_words = ['fàstic', 'depravat', 'depravada', 'malaltia', 'impur', 'impura', 'contagi', 'indecent', 'pecat', 'pecador', 'pecadora', 'pecats', 'pecant', 'puta', 'impietat', 'impiu', 'impia', 'profà', 'profana', 'brut', 'bruta', 'repugnant', 'malalt', 'malalta', 'promiscu', 'promiscua', 'adúlter', 'adultera', 'disbauxa', 'prostituta', 'prostitut', 'lasciu', 'brutícia', 'deshonrós', 'obscè', 'taca', 'tacar', 'degradar', 'profanar', 'malvat', 'malvada', 'explotar', 'pervertit', 'pervertida', 'miserable']

    # I didn't include no-binari, because it's two words and it doesn't have a vector
    lgbt_words = ['gay', 'lesbiana', 'bisexual', 'pansexual', 'transgènere', 'trans']

    disgust_vecs = np.array([embeddings[w] for w in disgust_words])
    lgbt_vecs = np.array([embeddings[w] for w in lgbt_words])

    #we construct a vector to represent the concept of moral disgust by averaging the vectors for all words in the “Moral Disgust” dictionary
    disgust_vec = np.mean(disgust_vecs, axis=0)

    #We also construct a vector to represent the concept of LGBT
    lgbt_vec = np.mean(lgbt_vecs, axis=0)

    #We measure the similarity
    #source = https://www.geeksforgeeks.org/python/how-to-calculate-cosine-similarity-in-python/
    cosine = np.dot(disgust_vec, lgbt_vec) / (norm(disgust_vec) * norm(lgbt_vec))
    print("Cosine Similarity for lgbt vector:", cosine)

    cosine = np.dot(disgust_vec, embeddings['fàstic']) / (norm(disgust_vec) * norm(embeddings['fàstic']))
    print("Cosine Similarity for the word 'fàstic':", cosine)

    cosine = np.dot(disgust_vec, embeddings['cadira']) / (norm(disgust_vec) * norm(embeddings['cadira']))
    print("Cosine Similarity for the word 'cadira':", cosine)

    cosine = np.dot(disgust_vec, embeddings['hetero']) / (norm(disgust_vec) * norm(embeddings['hetero'])) 
    print("Cosine Similarity for the word 'hetero':", cosine)

disgust_bias(floret_embeddings)

Cosine Similarity for lgbt vector: 0.35866332
Cosine Similarity for the word 'fàstic': 0.40927988
Cosine Similarity for the word 'cadira': 0.12919351
Cosine Similarity for the word 'hetero': 0.3079747


In [67]:
disgust_bias(ft_skipg_300)

Cosine Similarity for lgbt vector: 0.5026797
Cosine Similarity for the word 'fàstic': 0.6634092
Cosine Similarity for the word 'cadira': 0.32784432
Cosine Similarity for the word 'hetero': 0.48198053


In [72]:
#We conduct the Wilcoxon Signed-Rank test for the 200 neighbour valence values of two words: 
def wilcoxon_test(embeddings, score, target1, target2):
    print('-'*100)
    #We get all words but making sure that the words are in the NRC VAD Lexicon 
    all_words = list(embeddings.index_to_key) #these are the words in the embeddings
    catalan_words = set(df['Catalan']) #these are the words in the NRC VAD Lexicon for Catalan
    words_in_NRC = [w for w in all_words if w in catalan_words] #these are the match of both

    #We get the vectors 
    vecs_lexicon = np.array([embeddings[w] for w in words_in_NRC])

    # We start with just one target word 
    target_word1 = target1
    target_vec1 = embeddings[target_word1] 

    target_word2 = target2
    target_vec2 = embeddings[target_word2]  

    #source = https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html 
    # We get the nearest neighbors using similarity (replicating Mendelsohn, Tsvetkov and Jurafsky) using NearestNeighbors
    neigh = NearestNeighbors(n_neighbors=200, metric='cosine')
    neigh.fit(vecs_lexicon)


    #Here we compute the 200 nearest neighbors, we put the words in a list 
    neighbors1 = neigh.kneighbors([target_vec1], return_distance=False)[0]
    nearest_200_1 = [words_in_NRC[i] for i in neighbors1]

    neighbors2 = neigh.kneighbors([target_vec2], return_distance=False)[0]
    nearest_200_2 = [words_in_NRC[i] for i in neighbors2]

    if score == 'Valence':
        #Now we compute the mean average valence of all the 200 nearest neighbors 
        valence_1 = []
        for w in nearest_200_1:
            match = df[df['Catalan'] == w]
            if not match.empty:
                valence_1.append(match['Valence'].values[0])

        valence_2 = []
        for w in nearest_200_2:
            match = df[df['Catalan'] == w]
            if not match.empty:
                valence_2.append(match['Valence'].values[0])

        # conduct the Wilcoxon-Signed Rank Test
        print(f'{target1}: mean valence = {np.mean(valence_1):.4f}')
        print(f'{target2}: mean valence = {np.mean(valence_2):.4f}')
        print(stats.wilcoxon(valence_1, valence_2))

    if score == 'Dominance': 
        #Now we compute the mean average dominance of all the 200 nearest neighbors 
        dominance_1 = []
        for w in nearest_200_1:
            match = df[df['Catalan'] == w]
            if not match.empty:
                dominance_1.append(match['Dominance'].values[0])

        dominance_2 = []
        for w in nearest_200_2:
            match = df[df['Catalan'] == w]
            if not match.empty:
                dominance_2.append(match['Dominance'].values[0])

        # conduct the Wilcoxon-Signed Rank Test
        print(f'{target1}: mean dominance score = {np.mean(dominance_1):.4f}')
        print(f'{target2}: mean dominance score = {np.mean(dominance_2):.4f}')
        print(stats.wilcoxon(dominance_1, dominance_2))
    print('='*100)

In [92]:
wilcoxon_test(floret_embeddings,'Dominance','lesbiana', 'persona')

----------------------------------------------------------------------------------------------------
lesbiana: mean dominance score = 0.5349
persona: mean dominance score = 0.5640
WilcoxonResult(statistic=np.float64(8332.5), pvalue=np.float64(0.04675754877041295))
